# ROGII Submission Notebook

**Setup (run once in a terminal cell):**
```bash
cd /kaggle/working
git clone https://github.com/ibouftini/ROGII.git
pip install -q lightgbm xgboost catboost optuna tqdm
```

In [ ]:
# Inspect sample submission so we know exact expected format
import pandas as pd, os

COMP_DIR = '/kaggle/input/rogii-wellbore-geology-prediction'
sample = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')
print('=== SAMPLE SUBMISSION ===')
print(f'Shape : {sample.shape}')
print(f'Columns: {list(sample.columns)}')
print(f'Dtypes :\n{sample.dtypes}')
print(sample.head(3))

print('\n=== TEST FILES ===')
test_files = os.listdir(f'{COMP_DIR}/test')
print(f'{len(test_files)} files')
print(test_files[:4])

In [ ]:
import sys, os
import pandas as pd

REPO      = '/kaggle/working/ROGII'
COMP_DIR  = '/kaggle/input/rogii-wellbore-geology-prediction'
sys.path.insert(0, f'{REPO}/src')
sys.path.insert(0, REPO)

# Auto-detect models dir
MODELS_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.endswith('.pkl') for f in files):
        MODELS_DIR = root
        break
print(f'Models dir : {MODELS_DIR}')
assert MODELS_DIR, 'No .pkl files found — is rogii-models dataset attached?'

import config
from rogii.pipeline import run_pipeline

sub = run_pipeline(
    config,
    mode='predict',
    train_dir=f'{COMP_DIR}/train/',
    test_dir=f'{COMP_DIR}/test/',
    models_dir=MODELS_DIR,
)

assert sub is not None and not sub.empty, 'Empty predictions'

# Match sample submission column names exactly
sample = pd.read_csv(f'{COMP_DIR}/sample_submission.csv')
id_col  = sample.columns[0]   # whatever the competition calls it
tvt_col = sample.columns[1]
sub.columns = [id_col, tvt_col]

# Fill any NaN just in case
sub[tvt_col] = sub[tvt_col].fillna(sub[tvt_col].median())

print(f'Rows        : {len(sub):,}   (sample expects {len(sample):,})')
print(f'Columns     : {list(sub.columns)}')
print(f'NaN         : {sub.isna().sum().sum()}')
print(sub[tvt_col].describe())

sub.to_csv('submission.csv', index=False)
print('\nsubmission.csv saved')